# Test Suite Notebook – Phần 1

Notebook này được chỉnh lại theo **format gần giống `tests.ipynb`**:

1. Import Dependencies  
2. Implemented Methods  
3. Test Helpers  
4. Back Substitution Tests  
5. Gaussian Elimination Tests  
6. Determinant Tests  
7. Inverse Tests  
8. Rank and Basis Tests  
9. Stress Tests  
10. Run All Tests / Summary  

Mục tiêu của notebook:
- Giữ đúng các hàm bạn đã viết trong file `.py`
- **Nhúng trực tiếp các hàm kiểm chứng từ `verify.py`**
- Hạn chế kiểu “so sánh rời rạc” như trước, chuyển sang test theo nhóm bằng helper thống nhất
- Trình bày output theo kiểu rõ ràng hơn, giống notebook mẫu


## 1) Import Dependencies

In [1]:
import os
import sys
import random
import numpy as np

sys.path.append(os.getcwd())
sys.path.append("/mnt/data")

from determinant import determinant
from gaussian import gaussian_eliminate, back_substitution
from inverse import inverse

#Uu tien file rank_basis.py neu da doi ten theo cau truc nop bai
#neu chua co thi dung lai file rank_and_basic.py de tranh loi import
try:
    from rank_and_basis import rank_and_basis
except ImportError:
    from rank_and_basic import rank_and_basis

np.set_printoptions(precision=6, suppress=True)
EPS = 1e-9

## 2) Implemented Methods

In [2]:
implemented_methods = {
    "linear_system_methods": ["back_substitution", "gaussian_eliminate"],
    "matrix_methods": ["determinant", "inverse", "rank_and_basis"],
    "verification_methods_embedded_from_verify_py": [
        "verify_solution",
        "verify_inverse",
        "verify_determinant",
        "verify_rank",
        "verify_null_space",
        "verify_row_space",
        "verify_col_space",
    ],
}

for group, methods in implemented_methods.items():
    print(group + ":")
    for method in methods:
        print("  -", method)


linear_system_methods:
  - back_substitution
  - gaussian_eliminate
matrix_methods:
  - determinant
  - inverse
  - rank_and_basis
verification_methods_embedded_from_verify_py:
  - verify_solution
  - verify_inverse
  - verify_determinant
  - verify_rank
  - verify_null_space
  - verify_row_space
  - verify_col_space


## 3) Test Helpers

In [3]:


def verify_solution(A, x, b):
    return np.allclose(np.array(A, dtype=float) @ np.array(x, dtype=float),
                       np.array(b, dtype=float), atol=1e-5)

def verify_inverse(A, A_inv):
    if A_inv is None:
        return False
    A = np.array(A, dtype=float)
    A_inv = np.array(A_inv, dtype=float)
    I = np.eye(len(A))
    return np.allclose(A @ A_inv, I, atol=1e-5)

def verify_determinant(A, det_custom):
    det_np = np.linalg.det(np.array(A, dtype=float))
    return np.allclose(det_custom, det_np)

def verify_rank(A, rank_custom):
    rank_np = np.linalg.matrix_rank(np.array(A, dtype=float))
    return rank_custom == rank_np


def to_np(A):
    return np.array(A, dtype=float)

def fmt_matrix(M):
    return np.array(M, dtype=float)

def assert_true(cond, msg="assert_true failed"):
    if not cond:
        raise AssertionError(msg)

def assert_close(actual, expected, atol=1e-6, msg=""):
    a = np.array(actual, dtype=float)
    b = np.array(expected, dtype=float)
    if not np.allclose(a, b, atol=atol, rtol=0):
        raise AssertionError(f"{msg}\nactual={a}\nexpected={b}")

def verify_vector_close(x, y, atol=1e-6):
    return np.allclose(np.array(x, dtype=float), np.array(y, dtype=float), atol=atol, rtol=0)

def verify_matrix_close(A, B, atol=1e-6):
    return np.allclose(np.array(A, dtype=float), np.array(B, dtype=float), atol=atol, rtol=0)

def verify_scalar_close(a, b, atol=1e-6):
    return abs(float(a) - float(b)) <= atol

#Tinh residual ||Ax-b||
def residual_norm(A, x, b):
    return float(np.linalg.norm(to_np(A) @ to_np(x) - to_np(b)))

#Kiem tra U co dang tam giac tren hay khong
def verify_upper_triangular(U, atol=1e-10):
    U_np = np.array(U, dtype=float)
    return np.all(np.abs(np.tril(U_np, k=-1)) < atol)

#Kiem tra 1 vector co nam trong span cua 1 he vector hay khong
#as_columns=True  -> moi vector trong basis_vectors la 1 cot
#as_columns=False -> moi vector trong basis_vectors la 1 dong
def vector_in_span(v, basis_vectors, as_columns=True, atol=1e-7):
    v = np.array(v, dtype=float)

    if len(basis_vectors) == 0:
        return np.linalg.norm(v) <= atol

    B = np.array(basis_vectors, dtype=float)

    if as_columns:
        M = B
    else:
        M = B.T

    coef, *_ = np.linalg.lstsq(M, v, rcond=None)
    residual = np.linalg.norm(M @ coef - v)

    return residual <= atol * max(1.0, np.linalg.norm(v))

#Kiem tra null space chat hon:
#- dung so vector = nullity
#- cac vector doc lap tuyen tinh
#- moi vector deu thoa Ax=0
def verify_null_space(A, null_basis, atol=1e-8):
    A_np = np.array(A, dtype=float)
    n = A_np.shape[1]
    rank_np = np.linalg.matrix_rank(A_np)
    expected_nullity = n - rank_np

    if len(null_basis) != expected_nullity:
        return False

    if len(null_basis) == 0:
        return True

    B = np.array(null_basis, dtype=float)
    if np.linalg.matrix_rank(B) != len(null_basis):
        return False

    for v in null_basis:
        residual = np.linalg.norm(A_np @ np.array(v, dtype=float))
        if residual > atol * max(1.0, np.linalg.norm(v)):
            return False

    return True

#Kiem tra row space chat hon:
#- so vector dung bang rank
#- cac vector doc lap tuyen tinh
#- tung vector nam trong row space cua A
def verify_row_space(A, row_basis, atol=1e-7):
    A_np = np.array(A, dtype=float)
    rank_np = np.linalg.matrix_rank(A_np)
    n = A_np.shape[1]

    if len(row_basis) != rank_np:
        return False

    if any(len(v) != n for v in row_basis):
        return False

    if len(row_basis) == 0:
        return True

    B = np.array(row_basis, dtype=float)
    if np.linalg.matrix_rank(B) != len(row_basis):
        return False

    for v in row_basis:
        if not vector_in_span(v, A_np, as_columns=False, atol=atol):
            return False

    return True

#Kiem tra column space chat hon:
#- so vector dung bang rank
#- cac vector doc lap tuyen tinh
#- tung vector nam trong column space cua A
def verify_col_space(A, col_basis, atol=1e-7):
    A_np = np.array(A, dtype=float)
    rank_np = np.linalg.matrix_rank(A_np)
    m = A_np.shape[0]

    if len(col_basis) != rank_np:
        return False

    if any(len(v) != m for v in col_basis):
        return False

    if len(col_basis) == 0:
        return True

    B = np.column_stack([np.array(v, dtype=float) for v in col_basis])
    if np.linalg.matrix_rank(B) != len(col_basis):
        return False

    for v in col_basis:
        if not vector_in_span(v, A_np, as_columns=True, atol=atol):
            return False

    return True

def run_test(stats, name, fn, detail=True):
    stats["total"] += 1
    try:
        note = fn()
        stats["passed"] += 1
        print(f"[PASS]: {name}")
        if detail and note:
            print(note)
        print("_______________\n")
    except Exception as exc:
        print(f"[FAIL]: {name} -> {exc}")
        print("_______________\n")

def report(section, stats):
    print(f"[{section}] {stats['passed']}/{stats['total']} tests passed")
    return stats

def condition_number(A):
    try:
        return float(np.linalg.cond(np.array(A, dtype=float)))
    except Exception:
        return float("inf")

## 4) Back Substitution Tests

In [4]:
def test_back_substitution():
    stats = {"passed": 0, "total": 0}

    def t_upper_2x2():
        U = [[2, 3], [0, 4]]
        c = [8, 8]
        x = back_substitution(U, c)
        assert_true(verify_solution(U, x, c), "Ux != c")
        assert_true(verify_vector_close(x, [1.0, 2.0]), "expected [1,2]")
        assert_true(verify_upper_triangular(U), "U must be upper triangular")
        return f"U=\n{fmt_matrix(U)}\nc={c}\nx={np.array(x)}"

    run_test(stats, "back_substitution basic 2x2", t_upper_2x2)

    def t_identity():
        U = [[1,0,0],[0,1,0],[0,0,1]]
        c = [3.0, -1.0, 5.0]
        x = back_substitution(U, c)
        assert_true(verify_solution(U, x, c), "Ux != c")
        assert_true(verify_vector_close(x, c), "x must equal c for U=I")
        return f"U=I3\nc={c}\nx={np.array(x)}"

    run_test(stats, "back_substitution identity", t_identity)

    def t_upper_3x3():
        U = [[1, -2, 1], [0, 1, -1], [0, 0, 2]]
        c = [0, -1, 4]
        x = back_substitution(U, c)
        expected = np.linalg.solve(np.array(U, dtype=float), np.array(c, dtype=float))
        assert_true(verify_solution(U, x, c), "Ux != c")
        assert_close(x, expected, atol=1e-6, msg="3x3 mismatch")
        return f"U=\n{fmt_matrix(U)}\nc={c}\nx={np.array(x)}\nNumPy={expected}"

    run_test(stats, "back_substitution upper 3x3", t_upper_3x3)

    def t_float_matrix():
        U = [[3.5, 1.2, -0.8], [0, 2.1, 4.3], [0, 0, -1.7]]
        c = [4.1, 6.5, -3.4]
        x = back_substitution(U, c)
        assert_true(verify_solution(U, x, c), "float system verification failed")
        residual = residual_norm(U, x, c)
        assert_true(residual < 1e-6, f"residual too large: {residual}")
        return f"U=\n{fmt_matrix(U)}\nc={c}\nx={np.array(x)}\nresidual={residual:.3e}"

    run_test(stats, "back_substitution float coefficients", t_float_matrix)

    def t_random_4x4():
        np.random.seed(42)
        U = np.triu(np.random.randint(1, 10, (4,4))).tolist()
        c = [10.0, 5.0, 3.0, 1.0]
        x = back_substitution(U, c)
        assert_true(verify_solution(U, x, c), "random upper-triangular verification failed")
        assert_true(verify_upper_triangular(U), "generated U must be upper triangular")
        return f"U=\n{fmt_matrix(U)}\nc={c}\nx={np.array(x)}"

    run_test(stats, "back_substitution random 4x4", t_random_4x4)

    def t_1x1():
        U = [[7.0]]
        c = [21.0]
        x = back_substitution(U, c)
        assert_true(verify_solution(U, x, c), "1x1 system verification failed")
        assert_true(verify_scalar_close(x[0], 3.0), "x should be 3")
        return f"U={U}\nc={c}\nx={x}"

    run_test(stats, "back_substitution 1x1", t_1x1)

    def t_negative_coefficients():
        U = [[-2, 1, 3], [0, -4, 2], [0, 0, 5]]
        c = [7, -6, 10]
        x = back_substitution(U, c)
        expected = np.linalg.solve(to_np(U), to_np(c))
        assert_true(verify_solution(U, x, c), "negative-coefficient system failed")
        assert_close(x, expected, atol=1e-6, msg="negative coefficients mismatch")
        return f"U=\n{fmt_matrix(U)}\nc={c}\nx={np.array(x)}"

    run_test(stats, "back_substitution negative coefficients", t_negative_coefficients)

    return report("Back Substitution", stats)

back_substitution_stats = test_back_substitution()

[PASS]: back_substitution basic 2x2
U=
[[2. 3.]
 [0. 4.]]
c=[8, 8]
x=[1. 2.]
_______________

[PASS]: back_substitution identity
U=I3
c=[3.0, -1.0, 5.0]
x=[ 3. -1.  5.]
_______________

[PASS]: back_substitution upper 3x3
U=
[[ 1. -2.  1.]
 [ 0.  1. -1.]
 [ 0.  0.  2.]]
c=[0, -1, 4]
x=[0. 1. 2.]
NumPy=[0. 1. 2.]
_______________

[PASS]: back_substitution float coefficients
U=
[[ 3.5  1.2 -0.8]
 [ 0.   2.1  4.3]
 [ 0.   0.  -1.7]]
c=[4.1, 6.5, -3.4]
x=[ 1.971429 -1.        2.      ]
residual=0.000e+00
_______________

[PASS]: back_substitution random 4x4
U=
[[7. 4. 8. 5.]
 [0. 3. 7. 8.]
 [0. 0. 8. 8.]
 [0. 0. 0. 2.]]
c=[10.0, 5.0, 3.0, 1.0]
x=[ 0.857143  0.625    -0.125     0.5     ]
_______________

[PASS]: back_substitution 1x1
U=[[7.0]]
c=[21.0]
x=[3.0]
_______________

[PASS]: back_substitution negative coefficients
U=
[[-2.  1.  3.]
 [ 0. -4.  2.]
 [ 0.  0.  5.]]
c=[7, -6, 10]
x=[0.75 2.5  2.  ]
_______________

[Back Substitution] 7/7 tests passed


## 5) Gaussian Elimination Tests

In [5]:
def test_gaussian_eliminate():
    stats = {"passed": 0, "total": 0}

    def t_basic_2x2():
        A = [[2, 1], [5, 7]]
        b = [11, 13]
        U, x, swaps = gaussian_eliminate(A, b)
        assert_true(verify_solution(A, x, b), "Ax != b")
        assert_true(verify_upper_triangular(U), "U must be upper triangular")
        return f"A=\n{fmt_matrix(A)}\nb={b}\nU=\n{fmt_matrix(U)}\nx={np.array(x)}\nswaps={swaps}"

    run_test(stats, "gaussian basic 2x2", t_basic_2x2)

    def t_pivot_required():
        A = [[0, 1], [2, 3]]
        b = [1, 8]
        U, x, swaps = gaussian_eliminate(A, b)
        assert_true(verify_solution(A, x, b), "Ax != b after pivoting")
        assert_true(verify_upper_triangular(U), "U must be upper triangular")
        assert_true(swaps >= 1, "pivoting should swap at least once")
        return f"A=\n{fmt_matrix(A)}\nb={b}\nU=\n{fmt_matrix(U)}\nx={np.array(x)}\nswaps={swaps}"

    run_test(stats, "gaussian partial pivoting", t_pivot_required)

    def t_integer_3x3():
        A = [[1, 2, -1], [2, 1, 1], [-1, 1, 2]]
        b = [2, 7, 3]
        U, x, swaps = gaussian_eliminate(A, b)
        expected = np.linalg.solve(to_np(A), to_np(b))
        assert_true(verify_solution(A, x, b), "3x3 integer system failed")
        assert_true(verify_upper_triangular(U), "U must be upper triangular")
        assert_close(x, expected, atol=1e-6, msg="3x3 solution mismatch")
        return f"A=\n{fmt_matrix(A)}\nb={b}\nU=\n{fmt_matrix(U)}\nx={np.array(x)}\nNumPy={expected}\nswaps={swaps}"

    run_test(stats, "gaussian 3x3 integer solution", t_integer_3x3)

    def t_large_values():
        A = [[1e10, 2e10], [3e10, 4e10]]
        b = [3e10, 7e10]
        U, x, _ = gaussian_eliminate(A, b)
        assert_true(verify_solution(A, x, b), "large-value system failed")
        assert_true(verify_upper_triangular(U), "U must be upper triangular")
        return f"x={np.array(x)}\nresidual={residual_norm(A, x, b):.3e}"

    run_test(stats, "gaussian large values", t_large_values)

    def t_small_values():
        A = [[1e-10, 2e-10], [3e-10, 4e-10]]
        b = [3e-10, 7e-10]
        U, x, _ = gaussian_eliminate(A, b)
        assert_true(verify_solution(A, x, b), "small-value system failed")
        assert_true(verify_upper_triangular(U), "U must be upper triangular")
        return f"x={np.array(x)}\nresidual={residual_norm(A, x, b):.3e}"

    run_test(stats, "gaussian small values", t_small_values)

    def t_hilbert_5x5():
        A = [[1/(i+j+1) for j in range(5)] for i in range(5)]
        b = [sum(A[i]) for i in range(5)]
        U, x, _ = gaussian_eliminate(A, b)
        residual = residual_norm(A, x, b)
        assert_true(verify_solution(A, x, b), "Hilbert system failed")
        assert_true(verify_upper_triangular(U), "U must be upper triangular")
        return f"cond(A)={condition_number(A):.3e}\nresidual={residual:.3e}\nx={np.array(x)}"

    run_test(stats, "gaussian Hilbert 5x5", t_hilbert_5x5)

    def t_random_6x6():
        random.seed(0)
        A = [[random.random() for _ in range(6)] for _ in range(6)]
        b = [random.random() for _ in range(6)]
        U, x, _ = gaussian_eliminate(A, b)
        assert_true(verify_solution(A, x, b), "random 6x6 system failed")
        assert_true(verify_upper_triangular(U), "U must be upper triangular")
        return f"residual={residual_norm(A, x, b):.3e}"

    run_test(stats, "gaussian random 6x6", t_random_6x6)

    def t_compare_numpy_4x4():
        A = [[3, -1, 2, 4], [2, 5, 1, -3], [1, 0, 4, 2], [6, 1, -2, 1]]
        b = [10, 5, 7, 3]
        U, x, swaps = gaussian_eliminate(A, b)
        expected = np.linalg.solve(to_np(A), to_np(b))
        assert_true(verify_solution(A, x, b), "4x4 verification failed")
        assert_true(verify_upper_triangular(U), "U must be upper triangular")
        assert_close(x, expected, atol=1e-6, msg="4x4 mismatch with NumPy")
        return f"x={np.array(x)}\nNumPy={expected}\nswaps={swaps}"

    run_test(stats, "gaussian compare with NumPy 4x4", t_compare_numpy_4x4)

    def t_near_zero_first_pivot():
        A = [[1e-12, 1, 2], [1, 1, 3], [2, 4, 7]]
        b = [3, 5, 11]
        U, x, swaps = gaussian_eliminate(A, b)
        assert_true(verify_solution(A, x, b), "near-zero pivot case failed")
        assert_true(verify_upper_triangular(U), "U must be upper triangular")
        assert_true(swaps >= 1, "partial pivoting should swap in near-zero pivot case")
        return f"U=\n{fmt_matrix(U)}\nx={np.array(x)}\nswaps={swaps}"

    run_test(stats, "gaussian near-zero first pivot", t_near_zero_first_pivot)

    return report("Gaussian Elimination", stats)

gaussian_stats = test_gaussian_eliminate()

[PASS]: gaussian basic 2x2
A=
[[2. 1.]
 [5. 7.]]
b=[11, 13]
U=
[[ 5.   7. ]
 [ 0.  -1.8]]
x=[ 7.111111 -3.222222]
swaps=1
_______________

[PASS]: gaussian partial pivoting
A=
[[0. 1.]
 [2. 3.]]
b=[1, 8]
U=
[[2. 3.]
 [0. 1.]]
x=[2.5 1. ]
swaps=1
_______________

[PASS]: gaussian 3x3 integer solution
A=
[[ 1.  2. -1.]
 [ 2.  1.  1.]
 [-1.  1.  2.]]
b=[2, 7, 3]
U=
[[ 2.   1.   1. ]
 [ 0.   1.5 -1.5]
 [ 0.   0.   4. ]]
x=[2. 1. 2.]
NumPy=[2. 1. 2.]
swaps=1
_______________

[PASS]: gaussian large values
x=[1. 1.]
residual=0.000e+00
_______________

[PASS]: gaussian small values
x=[1. 1.]
residual=0.000e+00
_______________

[PASS]: gaussian Hilbert 5x5
cond(A)=4.766e+05
residual=5.207e-16
x=[1. 1. 1. 1. 1.]
_______________

[PASS]: gaussian random 6x6
residual=2.544e-16
_______________

[PASS]: gaussian compare with NumPy 4x4
x=[-0.906977  3.930233 -0.139535  4.232558]
NumPy=[-0.906977  3.930233 -0.139535  4.232558]
swaps=1
_______________

[PASS]: gaussian near-zero first pivot
U=
[[ 2.   

## 6) Determinant Tests

In [6]:
def test_determinant():
    stats = {"passed": 0, "total": 0}

    def t_1x1():
        A = [[7.0]]
        d = determinant(A)
        assert_true(verify_scalar_close(d, 7.0), "det([[7]]) should be 7")
        return f"A={A}\ndet={d}"

    run_test(stats, "determinant 1x1", t_1x1)

    def t_identity():
        A = [[1,0,0],[0,1,0],[0,0,1]]
        d = determinant(A)
        assert_true(verify_scalar_close(d, 1.0), "det(I) should be 1")
        return f"A=I3\ndet={d}"

    run_test(stats, "determinant identity", t_identity)

    def t_singular():
        A = [[1, 2], [2, 4]]
        d = determinant(A)
        assert_true(verify_scalar_close(d, 0.0, atol=1e-10), "singular matrix det should be 0")
        return f"A=\n{fmt_matrix(A)}\ndet={d}"

    run_test(stats, "determinant singular", t_singular)

    def t_2x2_formula():
        A = [[3, 8], [4, 6]]
        d = determinant(A)
        assert_true(verify_scalar_close(d, -14.0), "ad-bc mismatch")
        return f"A=\n{fmt_matrix(A)}\ndet={d}"

    run_test(stats, "determinant 2x2 ad-bc", t_2x2_formula)

    def t_sign_swap():
        A = [[0, 1, 2], [1, 0, 3], [4, 5, 6]]
        d = determinant(A)
        assert_true(verify_determinant(A, d), "determinant sign/value mismatch")
        return f"A=\n{fmt_matrix(A)}\ncustom det={d:.6f}\nNumPy det={np.linalg.det(to_np(A)):.6f}"

    run_test(stats, "determinant sign with swap", t_sign_swap)

    def t_general_3x3():
        A = [[2, -1, 3], [1, 4, -2], [5, 0, 1]]
        d = determinant(A)
        assert_true(verify_determinant(A, d), "general 3x3 determinant mismatch")
        return f"A=\n{fmt_matrix(A)}\ndet={d:.6f}\nNumPy det={np.linalg.det(to_np(A)):.6f}"

    run_test(stats, "determinant general 3x3", t_general_3x3)

    def t_random_5x5():
        random.seed(1)
        A = [[random.random() for _ in range(5)] for _ in range(5)]
        d = determinant(A)
        assert_true(verify_determinant(A, d), "random 5x5 determinant mismatch")
        return f"det={d:.6e}\nNumPy det={np.linalg.det(to_np(A)):.6e}"

    run_test(stats, "determinant random 5x5", t_random_5x5)

    def t_hilbert_4x4():
        A = [[1/(i+j+1) for j in range(4)] for i in range(4)]
        d = determinant(A)
        assert_true(verify_determinant(A, d), "Hilbert determinant mismatch")
        return f"det={d:.6e}\nNumPy det={np.linalg.det(to_np(A)):.6e}"

    run_test(stats, "determinant Hilbert 4x4", t_hilbert_4x4)

    return report("Determinant", stats)

determinant_stats = test_determinant()


[PASS]: determinant 1x1
A=[[7.0]]
det=7.0
_______________

[PASS]: determinant identity
A=I3
det=1.0
_______________

[PASS]: determinant singular
A=
[[1. 2.]
 [2. 4.]]
det=0.0
_______________

[PASS]: determinant 2x2 ad-bc
A=
[[3. 8.]
 [4. 6.]]
det=-14.0
_______________

[PASS]: determinant sign with swap
A=
[[0. 1. 2.]
 [1. 0. 3.]
 [4. 5. 6.]]
custom det=16.000000
NumPy det=16.000000
_______________

[PASS]: determinant general 3x3
A=
[[ 2. -1.  3.]
 [ 1.  4. -2.]
 [ 5.  0.  1.]]
det=-41.000000
NumPy det=-41.000000
_______________

[PASS]: determinant random 5x5
det=1.081322e-01
NumPy det=1.081322e-01
_______________

[PASS]: determinant Hilbert 4x4
det=1.653439e-07
NumPy det=1.653439e-07
_______________

[Determinant] 8/8 tests passed


## 7) Inverse Tests

In [7]:
def test_inverse():
    stats = {"passed": 0, "total": 0}

    def t_identity():
        A = [[1,0,0],[0,1,0],[0,0,1]]
        A_inv = inverse(A)
        assert_true(verify_inverse(A, A_inv), "I inverse verification failed")
        return f"A=I3\nA_inv=\n{fmt_matrix(A_inv)}"

    run_test(stats, "inverse identity", t_identity)

    def t_basic_2x2():
        A = [[4, 7], [2, 6]]
        A_inv = inverse(A)
        assert_true(verify_inverse(A, A_inv), "2x2 inverse failed")
        return f"A=\n{fmt_matrix(A)}\nA_inv=\n{fmt_matrix(A_inv)}"

    run_test(stats, "inverse basic 2x2", t_basic_2x2)

    def t_singular_none():
        A = [[1, 2], [2, 4]]
        A_inv = inverse(A)
        assert_true(A_inv is None, "singular matrix should return None")
        return f"A=\n{fmt_matrix(A)}\nA_inv={A_inv}"

    run_test(stats, "inverse singular returns None", t_singular_none)

    def t_general_3x3():
        A = [[2, 1, 1], [4, 3, 3], [8, 7, 9]]
        A_inv = inverse(A)
        assert_true(verify_inverse(A, A_inv), "general 3x3 inverse failed")
        return f"A_inv=\n{fmt_matrix(A_inv)}"

    run_test(stats, "inverse general 3x3", t_general_3x3)

    def t_diagonal():
        A = [[2,0,0],[0,3,0],[0,0,5]]
        A_inv = inverse(A)
        expected = [[0.5,0,0],[0,1/3,0],[0,0,0.2]]
        assert_true(verify_inverse(A, A_inv), "diagonal inverse verification failed")
        assert_true(verify_matrix_close(A_inv, expected), "diagonal inverse entries mismatch")
        return f"A_inv=\n{fmt_matrix(A_inv)}"

    run_test(stats, "inverse diagonal matrix", t_diagonal)

    def t_random_4x4():
        random.seed(5)
        A = [[random.uniform(1, 10) for _ in range(4)] for _ in range(4)]
        A_inv = inverse(A)
        assert_true(verify_inverse(A, A_inv), "random 4x4 inverse failed")
        return f"||AA_inv-I||_F={np.linalg.norm(to_np(A) @ to_np(A_inv) - np.eye(4)):.3e}"

    run_test(stats, "inverse random 4x4", t_random_4x4)

    def t_linear_dependent_columns():
        A = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
        A_inv = inverse(A)
        assert_true(A_inv is None, "dependent columns matrix should return None")
        return f"A=\n{fmt_matrix(A)}\nA_inv={A_inv}"

    run_test(stats, "inverse dependent columns returns None", t_linear_dependent_columns)

    return report("Inverse", stats)

inverse_stats = test_inverse()


[PASS]: inverse identity
A=I3
A_inv=
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
_______________

[PASS]: inverse basic 2x2
A=
[[4. 7.]
 [2. 6.]]
A_inv=
[[ 0.6 -0.7]
 [-0.2  0.4]]
_______________

[PASS]: inverse singular returns None
A=
[[1. 2.]
 [2. 4.]]
A_inv=None
_______________

[PASS]: inverse general 3x3
A_inv=
[[ 1.5 -0.5  0. ]
 [-3.   2.5 -0.5]
 [ 1.  -1.5  0.5]]
_______________

[PASS]: inverse diagonal matrix
A_inv=
[[0.5      0.       0.      ]
 [0.       0.333333 0.      ]
 [0.       0.       0.2     ]]
_______________

[PASS]: inverse random 4x4
||AA_inv-I||_F=1.972e-15
_______________

[PASS]: inverse dependent columns returns None
A=
[[1. 2. 3.]
 [4. 5. 6.]
 [7. 8. 9.]]
A_inv=None
_______________

[Inverse] 7/7 tests passed


## 8) Rank and Basis Tests

In [8]:
def test_rank_and_basis():
    stats = {"passed": 0, "total": 0}

    def t_identity():
        A = [[1,0,0],[0,1,0],[0,0,1]]
        rank, row_basis, col_basis, null_basis = rank_and_basis(A)
        assert_true(verify_rank(A, rank), "rank mismatch")
        assert_true(len(null_basis) == 0, "full-rank matrix should have empty null basis")
        assert_true(verify_row_space(A, row_basis), "row basis verification failed")
        assert_true(verify_col_space(A, col_basis), "col basis verification failed")
        return f"rank={rank}\nrow_basis={row_basis}\ncol_basis={col_basis}\nnull_basis={null_basis}"

    run_test(stats, "rank_and_basis identity", t_identity)

    def t_zero_matrix():
        A = [[0,0,0],[0,0,0],[0,0,0]]
        rank, row_basis, col_basis, null_basis = rank_and_basis(A)
        assert_true(rank == 0, "zero matrix rank should be 0")
        assert_true(len(row_basis) == 0, "zero matrix row basis should be empty")
        assert_true(len(col_basis) == 0, "zero matrix col basis should be empty")
        assert_true(len(null_basis) == 3, "null basis dimension should be 3")
        assert_true(verify_null_space(A, null_basis), "null basis invalid")
        return f"rank={rank}\nnull_basis={null_basis}"

    run_test(stats, "rank_and_basis zero matrix", t_zero_matrix)

    def t_dependent_rows():
        A = [[1, 2, 3], [2, 4, 6]]
        rank, row_basis, col_basis, null_basis = rank_and_basis(A)
        assert_true(verify_rank(A, rank), "rank mismatch")
        assert_true(verify_row_space(A, row_basis), "row basis verification failed")
        assert_true(verify_col_space(A, col_basis), "col basis verification failed")
        assert_true(len(null_basis) == 2, "nullity should be 2")
        assert_true(verify_null_space(A, null_basis), "invalid null basis")
        assert_true(rank + len(null_basis) == 3, "rank-nullity failed")
        return f"rank={rank}\nrow_basis={row_basis}\ncol_basis={col_basis}\nnull_basis={null_basis}"

    run_test(stats, "rank_and_basis dependent rows", t_dependent_rows)

    def t_rectangular_2x4():
        A = [[1, 2, 0, 3], [0, 0, 1, 4]]
        rank, row_basis, col_basis, null_basis = rank_and_basis(A)
        expected_cols = [[1, 0], [0, 1]]
        assert_true(verify_rank(A, rank), "rank mismatch")
        assert_true(verify_row_space(A, row_basis), "row basis verification failed")
        assert_true(verify_col_space(A, col_basis), "col basis verification failed")
        assert_true(verify_matrix_close(col_basis, expected_cols), "pivot columns from original matrix mismatch")
        assert_true(len(null_basis) == 2, "nullity should be 2")
        assert_true(verify_null_space(A, null_basis), "invalid null basis")
        return f"rank={rank}\ncol_basis={col_basis}\nnullity={len(null_basis)}\nnull_basis={null_basis}"

    run_test(stats, "rank_and_basis rectangular 2x4", t_rectangular_2x4)

    def t_square_rank_2():
        A = [[1,2,3],[4,5,6],[7,8,9]]
        rank, row_basis, col_basis, null_basis = rank_and_basis(A)
        assert_true(verify_rank(A, rank), "rank mismatch")
        assert_true(verify_row_space(A, row_basis), "row basis verification failed")
        assert_true(verify_col_space(A, col_basis), "col basis verification failed")
        assert_true(len(null_basis) == 1, "nullity should be 1")
        assert_true(verify_null_space(A, null_basis), "invalid null basis")
        assert_true(rank + len(null_basis) == 3, "rank-nullity failed")
        return f"rank={rank}\nrow_basis={row_basis}\ncol_basis={col_basis}\nnull_basis={null_basis}"

    run_test(stats, "rank_and_basis square rank 2", t_square_rank_2)

    def t_hidden_rank_4x4():
        A = [[1,2,3,4],[2,4,6,8],[3,5,7,9],[4,6,8,10]]
        rank, row_basis, col_basis, null_basis = rank_and_basis(A)
        assert_true(verify_rank(A, rank), "rank mismatch")
        assert_true(verify_row_space(A, row_basis), "row basis verification failed")
        assert_true(verify_col_space(A, col_basis), "col basis verification failed")
        assert_true(verify_null_space(A, null_basis), "invalid null basis")
        assert_true(rank + len(null_basis) == 4, "rank-nullity failed")
        return f"rank={rank}\nnullity={len(null_basis)}\nrank+nullity={rank + len(null_basis)}"

    run_test(stats, "rank_and_basis hidden rank 4x4", t_hidden_rank_4x4)

    def t_random_5x5():
        random.seed(10)
        A = [[random.randint(-3, 3) for _ in range(5)] for _ in range(5)]
        rank, row_basis, col_basis, null_basis = rank_and_basis(A)
        assert_true(verify_rank(A, rank), "rank mismatch")
        assert_true(verify_row_space(A, row_basis), "row basis verification failed")
        assert_true(verify_col_space(A, col_basis), "col basis verification failed")
        assert_true(verify_null_space(A, null_basis), "invalid null basis")
        assert_true(rank + len(null_basis) == 5, "rank-nullity failed")
        return f"A=\n{fmt_matrix(A)}\nrank={rank}\nnullity={len(null_basis)}"

    run_test(stats, "rank_and_basis random 5x5", t_random_5x5)

    def t_rectangular_3x5():
        A = [[1,2,3,4,5],[2,4,6,8,10],[0,1,1,1,1]]
        rank, row_basis, col_basis, null_basis = rank_and_basis(A)
        assert_true(verify_rank(A, rank), "rank mismatch")
        assert_true(verify_row_space(A, row_basis), "row basis verification failed")
        assert_true(verify_col_space(A, col_basis), "col basis verification failed")
        assert_true(verify_null_space(A, null_basis), "invalid null basis")
        assert_true(rank + len(null_basis) == 5, "rank-nullity failed")
        return f"rank={rank}\nrow_basis={row_basis}\nnull_basis={null_basis}"

    run_test(stats, "rank_and_basis rectangular 3x5", t_rectangular_3x5)

    return report("Rank and Basis", stats)

rank_basis_stats = test_rank_and_basis()

[PASS]: rank_and_basis identity
rank=3
row_basis=[[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
col_basis=[[1, 0, 0], [0, 1, 0], [0, 0, 1]]
null_basis=[]
_______________

[PASS]: rank_and_basis zero matrix
rank=0
null_basis=[[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
_______________

[PASS]: rank_and_basis dependent rows
rank=1
row_basis=[[1.0, 2.0, 3.0]]
col_basis=[[1, 2]]
null_basis=[[-2.0, 1.0, 0.0], [-3.0, 0.0, 1.0]]
_______________

[PASS]: rank_and_basis rectangular 2x4
rank=2
col_basis=[[1, 0], [0, 1]]
nullity=2
null_basis=[[-2.0, 1.0, 0.0, 0.0], [-3.0, 0.0, -4.0, 1.0]]
_______________

[PASS]: rank_and_basis square rank 2
rank=2
row_basis=[[1.0, 0.0, -0.9999999999999993], [0.0, 1.0, 1.9999999999999998]]
col_basis=[[1, 4, 7], [2, 5, 8]]
null_basis=[[0.9999999999999993, -1.9999999999999998, 1.0]]
_______________

[PASS]: rank_and_basis hidden rank 4x4
rank=2
nullity=2
rank+nullity=4
_______________

[PASS]: rank_and_basis random 5x5
A=
[[ 1. -3.  0.  0.  1.]
 [-3. -2